<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Allison/MLTestSuccess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import lightgbm as lgb


drive.mount('/content/drive')
path = "/content/drive/MyDrive/sparcs_cleaned_v3.csv"


# Columns needed for modeling
cols_needed = ['Length of Stay', 'Zip Code', 'Age Group', 'Gender', 'Race',
               'Ethnicity', 'Type of Admission', 'APR Risk of Mortality',
               'Medicare','Private Health Insurance','Self-Pay',
               'Blue Cross/Blue Shield','Miscellaneous/Other','Federal/State/Local/VA',
               'Department of Corrections','Managed Care, Unspecified','Number of Payment Typologies']

data = pd.read_csv(path, usecols=cols_needed)

# Convert categorical columns to category dtype
categorical_cols = ['Zip Code', 'Age Group', 'Gender', 'Race', 'Ethnicity',
                    'Type of Admission', 'APR Risk of Mortality']

for col in categorical_cols:
    data[col] = data[col].astype('category')

# Convert target to numeric and drop missing data
data['Length of Stay'] = pd.to_numeric(data['Length of Stay'], errors='coerce')
data = data.dropna(subset=['Length of Stay'])

# Remove out-of-state ZIP codes
if 'Zip Code' in data.columns:
    data = data[data['Zip Code'] != 'OOS']

# Numeric features
numeric_cols = ['Medicare','Private Health Insurance','Self-Pay',
                'Blue Cross/Blue Shield','Miscellaneous/Other','Federal/State/Local/VA',
                'Department of Corrections','Managed Care, Unspecified','Number of Payment Typologies']

# Define features and target
X = data[categorical_cols + numeric_cols]
y = data['Length of Stay']

# Column Transformer: sparse one-hot for categorical + passthrough numeric
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols)
    ],
    remainder='passthrough'
)

# LightGBM Regressor in Pipeline (memory-efficient and fast)
model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('regressor', lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=10,
        n_jobs=-1,
        random_state=42))])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.3f}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.403938 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 174
[LightGBM] [Info] Number of data points in the train set: 3285083, number of used features: 86
[LightGBM] [Info] Start training from score 5.664178


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MAE: 3.80
RMSE: 7.22
R2: 0.161
